# 05 — Building the unified modelling table

This notebook combines the common sample, task outcomes, approved control variables, representation features and diagnostic metadata into the single table used by all model experiments.

It does not fill missing values, standardise variables, encode categories, reduce dimensions or train a model. Those operations are learned separately within the training portion of each model evaluation. Keeping them out of the shared table prevents information from held-out locations influencing model preparation.

An explicit feature manifest defines which columns belong to each model specification. This is safer than selecting every numeric column because the source tables also contain fields created from the outcome for descriptive purposes.

## Summary of the output

The completed table contains 26,597 rows and 1,794 columns. Sample IDs are unique, both PTAL and EPC outcomes are complete, and all 33 borough groups are represented. The feature manifest records the 256 SatCLIP, 128 TESSERA, 64 AlphaEarth, 768 DINOv2 and 512 Street View dimensions, together with the approved control and combined feature sets.

Outcome-derived rating bands, alternative outcomes and corrected-versus-earlier differences are excluded from model inputs. Street View visual values remain missing where imagery is unavailable, allowing the model pipeline to handle them using training data only.

In [ ]:
# Connect Google Drive and load the common project configuration.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import gc
import numpy as np
import pandas as pd

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

required = {
    "common sample": COMMON_SAMPLE_FINAL_PATH,
    "SatCLIP": SATCLIP_EMB_PATH,
    "TESSERA": TESSERA_EMB_PATH,
    "AlphaEarth final": ALPHA_FINAL_PATH,
    "DINOv2": DINO2_EMB_PATH,
    "Street View": STREET_EMB_PATH,
}
for name, path in required.items():
    print(f"{name:20s} | {path.exists()} | {path}")

missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required project inputs: {missing}")

## 1. Create the model-facing base table

Only stable identifiers, geography, the continuous outcome, approved controls, diagnostic metadata and representation features are retained. Fields that directly restate or are derived from the target are excluded from the model-facing table.

In [ ]:
# Create the model-facing base table from stable IDs, geography and outcomes.
common = pd.read_parquet(COMMON_SAMPLE_FINAL_PATH).copy()
common["sample_id"] = common["sample_id"].astype(str)
common["task"] = common["task"].astype(str).str.upper()

assert len(common) == 26597
assert common["sample_id"].is_unique
assert common["label_regression"].notna().all()
assert common["borough_code"].notna().all()
assert common["borough_code"].nunique() == 33

# Stable metadata only.
metadata_candidates = [
    "sample_id", "task",
    "x", "y", "lon", "lat",
    "borough", "borough_code",
    "postcode_clean",
]
metadata_cols = [c for c in metadata_candidates if c in common.columns]

model = common[metadata_cols].copy()
model["target"] = pd.to_numeric(common["label_regression"], errors="coerce")

assert model["target"].notna().all()

# Explicitly derive a small, interpretable spatial baseline.
model["x_km"] = (pd.to_numeric(common["x"]) - LONDON_CENTRE_EASTING) / 1000.0
model["y_km"] = (pd.to_numeric(common["y"]) - LONDON_CENTRE_NORTHING) / 1000.0
model["x_km2"] = model["x_km"] ** 2
model["y_km2"] = model["y_km"] ** 2
model["xy_km2"] = model["x_km"] * model["y_km"]
model["dist_centre_km"] = np.sqrt(model["x_km"] ** 2 + model["y_km"] ** 2)

display(model.head())
print("Base table shape:", model.shape)

## 2. Add EPC control variables

For PTAL, coordinate terms form a simple location baseline. For EPC, the compact control set combines the spatial trend with postcode property count, modal property type and modal built form. The richer EPC control set additionally includes median floor area and modal construction-age band.

Record dates, completeness shares and composition fields remain available for descriptive or supplementary work but are not silently included in the main control specification.

In [ ]:
# Attach only approved EPC control and diagnostic variables.
epc_safe_candidates = [
    "n_properties",
    "property_type",
    "built_form",
    "median_total_floor_area",
    "mean_total_floor_area",
    "construction_age_band_mode",
    "floor_area_valid_share",
    "age_valid_share",
    "share_uprn",
    "reliable_n3",
    "min_record_date",
    "median_record_date",
    "max_record_date",
]

epc_safe_cols = [c for c in epc_safe_candidates if c in common.columns]

# Optional composition variables are safe descriptors, but are reserved for robustness.
composition_cols = [
    c for c in common.columns
    if c.startswith("share_age_")
    or c.startswith("share_property_type_")
    or c.startswith("share_built_form_")
]

for c in epc_safe_cols + composition_cols:
    model[c] = common[c]

if "n_properties" in model.columns:
    nprop = pd.to_numeric(model["n_properties"], errors="coerce")
    # Every sampled EPC postcode contains at least one retained property.\n    # Use the conventional natural-log count transform; no +1 offset is needed.\n    epc_mask = model["task"].eq("EPC")\n    assert (nprop.loc[epc_mask] > 0).all(), "EPC n_properties must be positive."\n    model["log_n_properties"] = np.log(nprop.where(nprop > 0))

for c in ["min_record_date", "median_record_date", "max_record_date"]:
    if c in model.columns:
        model[c] = pd.to_datetime(model[c], errors="coerce")

if "median_record_date" in model.columns:
    d = model["median_record_date"]
    model["median_record_year"] = np.where(
        d.notna(),
        d.dt.year + (d.dt.dayofyear - 1) / 365.25,
        np.nan
    )

print("Approved EPC summary/control columns:", epc_safe_cols)
print("Optional composition columns:", len(composition_cols))

## 3. Add aerial acquisition metadata

The source-tile information derived in Notebook 04 is attached to each DINOv2 crop. These fields allow imagery-year diagnostics but are not part of the main DINOv2 representation.

In [ ]:
# Attach aerial acquisition metadata for diagnostics, not primary prediction.
if AERIAL_SAMPLE_TEMPORAL_AUDIT_PATH.exists():
    aerial_time = pd.read_csv(AERIAL_SAMPLE_TEMPORAL_AUDIT_PATH)
    aerial_time["sample_id"] = aerial_time["sample_id"].astype(str)

    keep_aerial_time = [
        "sample_id",
        "aerial_source_type",
        "aerial_n_source_tiles",
        "aerial_date_min",
        "aerial_date_max",
        "aerial_year_min",
        "aerial_year_max",
        "aerial_mixed_years",
        "aerial_date_span_days",
    ]
    aerial_time = aerial_time[keep_aerial_time].copy()

    for c in ["aerial_date_min", "aerial_date_max"]:
        aerial_time[c] = pd.to_datetime(aerial_time[c], errors="coerce")

    model = model.merge(
        aerial_time,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    assert model["aerial_year_min"].notna().all()
    print("Aerial sample-level temporal metadata merged.")
else:
    print(
        "WARNING: sample-level aerial temporal audit not found. "
        "The model table can still be built, but the temporal diagnostic "
        "metadata will be absent."
    )

## 4. Merge the five representation families

SatCLIP, TESSERA, AlphaEarth, DINOv2 and Street View are joined one-to-one by `sample_id`. For locations without suitable Street View imagery, all 512 visual dimensions remain missing and the availability metadata record the reason.

In [ ]:
# Join each representation one-to-one by sample ID.
representation_specs = {
    "SatCLIP": {
        "path": SATCLIP_EMB_PATH,
        "prefix": "satclip_",
        "dim": 256,
    },
    "TESSERA": {
        "path": TESSERA_EMB_PATH,
        "prefix": "tessera_",
        "dim": 128,
    },
    "AlphaEarth": {
        "path": ALPHA_FINAL_PATH,
        "prefix": "alphaearth_2024_",
        "dim": 64,
    },
    "DINOv2": {
        "path": DINO2_EMB_PATH,
        "prefix": "dinov2_",
        "dim": 768,
    },
    "StreetView_CLIP": {
        "path": STREET_EMB_PATH,
        "prefix": "streetclip_agg_",
        "dim": 512,
    },
}

feature_groups = {}
merge_audit = []

for name, spec in representation_specs.items():
    emb = pd.read_parquet(spec["path"]).copy()
    emb["sample_id"] = emb["sample_id"].astype(str)

    assert not emb["sample_id"].duplicated().any(), f"{name}: duplicate sample IDs"

    feats = [c for c in emb.columns if str(c).startswith(spec["prefix"])]
    assert len(feats) == spec["dim"], (
        f"{name}: detected {len(feats)} features; expected {spec['dim']}"
    )

    common_ids = set(model["sample_id"])
    emb_ids = set(emb["sample_id"])
    assert common_ids.issubset(emb_ids), f"{name}: missing common-sample IDs"

    keep = ["sample_id"] + feats

    if name == "StreetView_CLIP":
        sv_meta = [
            c for c in [
                "sv_has_streetview",
                "sv_n_images",
                "sv_min_dist_m",
                "sv_mean_dist_m",
            ]
            if c in emb.columns
        ]
        assert len(sv_meta) == 4, f"Street View metadata incomplete: {sv_meta}"
        keep += sv_meta
        feature_groups["streetview_metadata"] = sv_meta

    sub = emb.loc[emb["sample_id"].isin(common_ids), keep].copy()
    assert len(sub) == len(model), f"{name}: unexpected common-sample row count"

    # Float32 is sufficient for pretrained embeddings and avoids an unnecessarily
    # large canonical Parquet file.
    sub[feats] = sub[feats].astype("float32")

    model = model.merge(
        sub,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )

    complete_pct = model[feats].notna().all(axis=1).mean() * 100
    merge_audit.append({
        "representation": name,
        "n_features": len(feats),
        "complete_rows_pct": float(complete_pct),
    })
    feature_groups[name.lower()] = feats

    del emb, sub
    gc.collect()

merge_audit = pd.DataFrame(merge_audit)
display(merge_audit)

## 5. Define the model specifications

Each named feature set is written as an ordered list of permitted columns. The definitions include individual representations, Street View content and metadata variants, theory-informed compact combinations, all-representation combinations, and task-specific control sets. These lists are the common interface used by every model notebook.

In [ ]:
# Define every individual, combined and control feature set explicitly.
spatial_controls = [
    "x_km", "y_km", "x_km2", "y_km2", "xy_km2", "dist_centre_km"
]

epc_sparse_extra = [
    c for c in ["log_n_properties", "property_type", "built_form"]
    if c in model.columns
]

epc_extensive_extra = [
    c for c in ["median_total_floor_area", "construction_age_band_mode"]
    if c in model.columns
]

epc_temporal_robustness_extra = [
    c for c in ["median_record_year"]
    if c in model.columns
]

epc_quality_metadata = [
    c for c in [
        "n_properties", "reliable_n3", "share_uprn",
        "floor_area_valid_share", "age_valid_share",
        "min_record_date", "median_record_date", "max_record_date",
        "median_record_year",
    ]
    if c in model.columns
]

feature_sets = {
    "PTAL_spatial_baseline": spatial_controls,
    "EPC_controls_sparse": spatial_controls + epc_sparse_extra,
    "EPC_controls_extensive": (
        spatial_controls + epc_sparse_extra + epc_extensive_extra
    ),
    "EPC_controls_extensive_plus_record_year_robustness": (
        spatial_controls + epc_sparse_extra
        + epc_extensive_extra + epc_temporal_robustness_extra
    ),
    "SatCLIP": feature_groups["satclip"],
    "TESSERA": feature_groups["tessera"],
    "AlphaEarth": feature_groups["alphaearth"],
    "DINOv2": feature_groups["dinov2"],
    "StreetView_CLIP_only": feature_groups["streetview_clip"],
    "StreetView_metadata_only": feature_groups["streetview_metadata"],
    "StreetView_CLIP_plus_metadata": (
        feature_groups["streetview_clip"]
        + feature_groups["streetview_metadata"]
    ),
}

feature_sets["All_representations_without_SV_metadata"] = (
    feature_sets["SatCLIP"]
    + feature_sets["TESSERA"]
    + feature_sets["AlphaEarth"]
    + feature_sets["DINOv2"]
    + feature_sets["StreetView_CLIP_only"]
)

feature_sets["All_representations_plus_SV_metadata"] = (
    feature_sets["All_representations_without_SV_metadata"]
    + feature_sets["StreetView_metadata_only"]
)

# A few scientifically interpretable fusion groups.
feature_sets["Sky_and_space"] = (
    feature_sets["DINOv2"]
    + feature_sets["TESSERA"]
    + feature_sets["AlphaEarth"]
)
feature_sets["Street_and_sky"] = (
    feature_sets["StreetView_CLIP_only"]
    + feature_sets["DINOv2"]
)
feature_sets["Location_and_EO"] = (
    feature_sets["SatCLIP"]
    + feature_sets["TESSERA"]
    + feature_sets["AlphaEarth"]
)

# Sanity checks.
assert len(feature_sets["SatCLIP"]) == 256
assert len(feature_sets["TESSERA"]) == 128
assert len(feature_sets["AlphaEarth"]) == 64
assert len(feature_sets["DINOv2"]) == 768
assert len(feature_sets["StreetView_CLIP_only"]) == 512
assert len(feature_sets["All_representations_without_SV_metadata"]) == 1728

print("Feature-set sizes")
display(
    pd.Series(
        {k: len(v) for k, v in feature_sets.items()},
        name="n_features"
    ).sort_values()
)

## 6. Exclude outcome information from predictors

The combined source data contain alternative outcomes, rating classes and comparison fields that are useful for data preparation but would reveal the answer to a model. This section confirms that none of these fields appears in any declared predictor set.

In [ ]:
# Confirm that target-derived fields cannot enter any predictor set.
forbidden_exact = {
    "label_regression",
    "label_classification",
    "current_energy_efficiency",
    "current_energy_rating",
    "epc_level",
    "ptal_level",
    "uprn_only_target",
    "full_minus_uprn_only_target",
    "target_diff",
    "label_regression_legacy",
    "label_classification_legacy",
    "epc_level_legacy",
}

forbidden_substrings = [
    "_legacy",
    "target_diff",
    "uprn_only_target",
    "full_minus_uprn_only_target",
]

leak_exact_present = sorted(forbidden_exact.intersection(model.columns))
leak_pattern_present = sorted({
    c for c in model.columns
    if any(token in c for token in forbidden_substrings)
})

assert not leak_exact_present, (
    f"Forbidden target-derived columns present: {leak_exact_present}"
)
assert not leak_pattern_present, (
    f"Forbidden leakage-pattern columns present: {leak_pattern_present}"
)

# Ensure the only model outcome column is the explicit target.
assert "target" in model.columns
print("Leakage guard: PASS")

## 7. Describe missing values before model fitting

Missingness is reported separately for PTAL and EPC. Street View dimensions are expected to be jointly present or absent for a location, and their pattern must agree with the availability flag. Other missing values remain untouched so that their treatment can be learned from training data during evaluation.

In [ ]:
# Describe missing values by task before any training-based treatment.
missing_rows = []

audit_groups = {
    "spatial_controls": spatial_controls,
    "epc_sparse_extra": epc_sparse_extra,
    "epc_extensive_extra": epc_extensive_extra,
    "satclip": feature_sets["SatCLIP"],
    "tessera": feature_sets["TESSERA"],
    "alphaearth": feature_sets["AlphaEarth"],
    "dinov2": feature_sets["DINOv2"],
    "streetview_clip": feature_sets["StreetView_CLIP_only"],
    "streetview_metadata": feature_sets["StreetView_metadata_only"],
}

for task, task_df in model.groupby("task"):
    for group_name, cols in audit_groups.items():
        if not cols:
            continue
        complete = task_df[cols].notna().all(axis=1)
        missing_rows.append({
            "task": task,
            "feature_group": group_name,
            "n_rows": int(len(task_df)),
            "n_complete_rows": int(complete.sum()),
            "complete_rows_pct": float(complete.mean() * 100),
        })

missingness = pd.DataFrame(missing_rows)
display(missingness)
missingness.to_csv(FINAL_MODEL_MISSINGNESS_PATH, index=False)

sv_cols = feature_sets["StreetView_CLIP_only"]
sv_complete = model[sv_cols].notna().all(axis=1)
sv_all_missing = model[sv_cols].isna().all(axis=1)
assert (sv_complete | sv_all_missing).all(), (
    "Street View contains partially missing visual vectors."
)

sv_flag = (
    pd.to_numeric(model["sv_has_streetview"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(bool)
)
assert (sv_complete == sv_flag).all(), (
    "Street View CLIP completeness disagrees with availability flag."
)

## 8. Save the feature manifest

The JSON manifest is read directly by the model code, while a long CSV version makes every model input easy to inspect. Categorical variables are stored as their original labels and are encoded only during training.

In [ ]:
# Write machine-readable and human-readable versions of the feature manifest.
manifest = {
    "target_column": "target",
    "id_column": "sample_id",
    "task_column": "task",
    "group_column": "borough_code",
    "coordinate_columns": ["x", "y", "lon", "lat"],
    "feature_sets": feature_sets,
    "epc_quality_metadata": epc_quality_metadata,
    "aerial_temporal_metadata": [
        c for c in [
            "aerial_source_type", "aerial_n_source_tiles",
            "aerial_date_min", "aerial_date_max",
            "aerial_year_min", "aerial_year_max",
            "aerial_mixed_years", "aerial_date_span_days",
        ] if c in model.columns
    ],
    "optional_epc_composition_features": composition_cols,
    "categorical_columns": [
        c for c in [
            "property_type",
            "built_form",
            "construction_age_band_mode",
        ]
        if c in model.columns
    ],
    "rules": {
        "imputation": "fit inside each training fold only",
        "scaling": "fit inside each training fold only",
        "categorical_encoding": "fit inside each training fold only",
        "target_derived_fields": "excluded from canonical model table",
        "streetview_missing_vectors": "preserved as NaN at table-construction stage",
    },
}

with open(FEATURE_MANIFEST_JSON_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

manifest_rows = []
for set_name, cols in feature_sets.items():
    for position, col in enumerate(cols):
        manifest_rows.append({
            "feature_set": set_name,
            "position": position,
            "column": col,
        })

pd.DataFrame(manifest_rows).to_csv(
    FEATURE_MANIFEST_CSV_PATH,
    index=False
)

print("Saved:", FEATURE_MANIFEST_JSON_PATH)
print("Saved:", FEATURE_MANIFEST_CSV_PATH)

## 9. Save the canonical table

The final Parquet table and summary are saved after confirming row counts, task counts, borough coverage, target completeness, unique sample IDs, declared feature dimensions and the absence of prohibited predictors. The resulting table is the fixed data source for the comparative experiments.

In [ ]:
# Save the canonical table and its structural summary.
# Preserve date columns as timestamps; Parquet stores them safely.
model.to_parquet(FINAL_MODEL_TABLE_PATH, index=False)

audit_summary = {
    "n_rows": int(len(model)),
    "n_columns": int(model.shape[1]),
    "n_epc": int((model["task"] == "EPC").sum()),
    "n_ptal": int((model["task"] == "PTAL").sum()),
    "n_borough_groups": int(model["borough_code"].nunique()),
    "sample_id_unique": bool(model["sample_id"].is_unique),
    "target_complete_pct": float(model["target"].notna().mean() * 100),
    "n_satclip_features": len(feature_sets["SatCLIP"]),
    "n_tessera_features": len(feature_sets["TESSERA"]),
    "n_alphaearth_features": len(feature_sets["AlphaEarth"]),
    "n_dinov2_features": len(feature_sets["DINOv2"]),
    "n_streetview_clip_features": len(feature_sets["StreetView_CLIP_only"]),
    "n_all_representation_features_without_sv_metadata": len(
        feature_sets["All_representations_without_SV_metadata"]
    ),
    "streetview_clip_complete_pct": float(sv_complete.mean() * 100),
    "leakage_guard_pass": True,
    "preprocessing_applied_globally": False,
}

assert audit_summary["n_rows"] == 26597
assert audit_summary["n_epc"] == 20000
assert audit_summary["n_ptal"] == 6597
assert audit_summary["n_borough_groups"] == 33
assert audit_summary["sample_id_unique"]
assert np.isclose(audit_summary["target_complete_pct"], 100.0)
assert audit_summary["n_all_representation_features_without_sv_metadata"] == 1728
assert audit_summary["leakage_guard_pass"]
assert not audit_summary["preprocessing_applied_globally"]

with open(FINAL_MODEL_AUDIT_SUMMARY_PATH, "w") as f:
    json.dump(audit_summary, f, indent=2)

print("05 unified model-table decision gate: PASS")
display(pd.Series(audit_summary, name="value"))
print("Saved:", FINAL_MODEL_TABLE_PATH)
print("Saved:", FINAL_MODEL_AUDIT_SUMMARY_PATH)